# `data.py` en acción — construcción de datos para WR semanal

## Qué es esto y en qué se diferencia del notebook anterior

[`1.1_calidad_fuentes.ipynb`](1.1_calidad_fuentes.ipynb) respondió una pregunta: **¿se puede confiar en
`nflreadpy` como fuente?** Esa auditoría ya está hecha y documentada.

Este notebook responde una pregunta distinta: **¿qué hace el código real del pipeline
(`weekly/wr/src/data.py`) y por qué está construido así?** No es una auditoría: muestra las funciones
de carga con datos reales y explica algunas decisiones que no son obvias con solo leer el código. Las
columnas que aparecen se definen en el [glosario de variables](../../../docs/data/GLOSARIO_VARIABLES.md).

Siguiendo la [regla de escalabilidad](../../../docs/PLAN.md) del proyecto, la lógica real vive en
`.py` (`data.py`) y este notebook solo la importa y la muestra, sin duplicar lógica.

## Las funciones de carga de `data.py`

Todas consultan `nflreadpy` directamente. Las de calendario, estadísticas y jugadas ya filtran a
temporada regular.

| Función | Qué trae | Para qué se usa |
|---|---|---|
| `cargar_jugadores()` | Identidad de cada jugador histórico | Cruzar `gsis_id` con las demás fuentes, sin los problemas del archivo de ADP (ver la Fuente 1 de `1.1_calidad_fuentes.ipynb`) |
| `cargar_calendario(seasons)` | Partidos de temporada regular | Fecha de cada partido y contexto (apuestas, clima) |
| `cargar_stats_semanales(seasons)` | Lo que produjo cada jugador, cada semana | Es lo que se quiere predecir (recepciones, yardas, touchdowns) |
| `cargar_jugadas(seasons)` | Detalle de cada jugada | Calcular agregados que la tabla semanal no trae |
| `cargar_depth_charts_historico(seasons)` | Alineación titular, temporadas ≤2024 | Variables de entrenamiento histórico |
| `cargar_depth_charts_actual(seasons)` | Alineación titular, 2025 en adelante (incluida la temporada en curso) | Saber quién es titular para la predicción de la semana en curso |

In [1]:
import sys
sys.path.insert(0, "../src")
import data

SEASONS = [2025]  # temporada de ejemplo para esta demostracion

### `cargar_jugadores()`

No recibe temporada porque es la tabla maestra de todos los jugadores que ha habido en la
liga.

In [2]:
jugadores = data.cargar_jugadores()
print("filas:", jugadores.shape[0], "| columnas:", jugadores.shape[1])
print("dtype de birth_date (ya convertido a fecha real, no texto):", jugadores["birth_date"].dtype)
jugadores[["gsis_id", "display_name", "position", "birth_date"]].head(3)

filas: 24834 | columnas: 39
dtype de birth_date (ya convertido a fecha real, no texto): datetime64[ns]


,gsis_id,display_name,position,birth_date
0,00-0028830,Isaako Aaitui,NT,1987-01-25
1,00-0038389,Israel Abanikanda,RB,2002-10-05
2,00-0024644,Jon Abbate,LB,1985-06-18


### `cargar_calendario(seasons)`

Una fila por partido, con fecha, equipos, línea de apuestas y clima.

In [3]:
calendario = data.cargar_calendario(SEASONS)
print("filas:", calendario.shape[0], "| semanas:", sorted(calendario["week"].unique()))
calendario[["season", "week", "gameday", "home_team", "away_team", "spread_line"]].head(3)

filas: 272 | semanas: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18)]


,season,week,gameday,home_team,away_team,spread_line
0,2025,1,2025-09-04,PHI,DAL,8.5
1,2025,1,2025-09-05,LAC,KC,-3.0
2,2025,1,2025-09-07,ATL,TB,-1.5


### `cargar_stats_semanales(seasons)`

Una fila por jugador y partido: es la tabla de la que sale lo que se quiere predecir.

In [4]:
stats = data.cargar_stats_semanales(SEASONS)
wr = stats[stats["position"] == "WR"]
print("filas totales:", stats.shape[0], "| filas WR:", wr.shape[0])
wr[["player_display_name", "week", "receptions", "receiving_yards", "receiving_tds", "target_share"]].head(3)

filas totales: 18540 | filas WR: 2511


,player_display_name,week,receptions,receiving_yards,receiving_tds,target_share
23,Adam Thielen,1,0,0,0,0.050000
28,Keenan Allen,1,7,68,1,0.294118
30,DeAndre Hopkins,1,2,35,1,0.105263


### `cargar_jugadas(seasons)`

Detalle de cada jugada. No se lee directo: sirve para calcular agregados que la tabla semanal no
trae. `target_share`, que en los notebooks originales se calculaba desde aquí, ya viene en
`cargar_stats_semanales` (ver `1.1_calidad_fuentes.ipynb`).

In [5]:
jugadas = data.cargar_jugadas(SEASONS)
print("filas (una por jugada):", jugadas.shape[0])
jugadas[["game_id", "week", "receiver_player_name", "pass_attempt", "air_yards"]].dropna(subset=["receiver_player_name"]).head(3)

filas (una por jugada): 46452


,game_id,week,receiver_player_name,pass_attempt,air_yards
3,2025_01_ARI_NO,1,T.McBride,1.0,3.0
9,2025_01_ARI_NO,1,R.Shaheed,1.0,2.0
12,2025_01_ARI_NO,1,T.McBride,1.0,-3.0


### `cargar_depth_charts_historico(seasons)` y `cargar_depth_charts_actual(seasons)`

Estas dos son las más delicadas porque hay que saber exactamente qué filtrar. `nflreadpy` cambia de
esquema entre las temporadas ≤2024 y 2025 en adelante (ver la Fuente 5 de
`1.1_calidad_fuentes.ipynb`), y **ninguno de los dos esquemas se puede filtrar solo por el nombre obvio
de columna** sin producir un resultado incorrecto y sin ningún error visible. Cada trampa se muestra
con el dato real que la reveló. Qué es una alineación y qué significan sus columnas está en el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md) (sección 1 y apéndice B.3).

**Trampa 1 (esquema viejo): `position` sola no basta.** Un jugador puede aparecer con
`position="WR"` también en una fila de *Special Teams* (por ejemplo, como regresador de despejes), que
no es su rol de receptor. Ejemplo: Britain Covey (PHI), semana 1 de 2024.

In [6]:
import nflreadpy as nfl
d24 = nfl.load_depth_charts([2024])
covey = d24.filter((d24["full_name"] == "Britain Covey") & (d24["week"] == 1))
covey.select(["club_code", "week", "formation", "depth_position", "position", "depth_team", "full_name"])

club_code,week,formation,depth_position,position,depth_team,full_name
str,i32,str,str,str,str,str
"""PHI""",1,"""Special Teams""","""PR""","""WR""","""1""","""Britain Covey"""
"""PHI""",1,"""Offense""","""WR""","""WR""","""2""","""Britain Covey"""


Aparece dos veces: como receptor de ataque (`formation="Offense"`, `depth_position="WR"`) y como
regresador de despejes (`formation="Special Teams"`, `depth_position="PR"`), con `position="WR"` en
ambas filas. Por eso `cargar_depth_charts_historico()` filtra por `formation` y `depth_position`, no
solo por `position`.

**Trampa 2 (esquema nuevo): `pos_grp` no es la posición del jugador.** Son paquetes de
formación.

In [7]:
d26 = nfl.load_depth_charts([2026])
print("valores de pos_grp (NO son posiciones):", sorted(d26["pos_grp"].unique().to_list()))
print("la posicion real vive en pos_name:", sorted(v for v in d26["pos_name"].unique().to_list() if v and "Wide" in v))

valores de pos_grp (NO son posiciones): ['3WR 1TE', 'Base 3-4 D', 'Base 4-3 D', 'Special Teams']
la posicion real vive en pos_name: ['Wide Receiver']


Por eso `cargar_depth_charts_actual()` filtra por `pos_abb` (equivalente abreviado de
`pos_name`), no por `pos_grp`.

**Trampa 3 (esquema viejo): existe una "semana 19" etiquetada `REG`.** Confirmado con un equipo
que no jugó playoffs — no puede ser un partido real.

In [8]:
atl_19 = d24.filter((d24["club_code"] == "ATL") & (d24["week"] == 19) & (d24["game_type"] == "REG"))
print("filas de ATL en 'semana 19' marcada REG:", atl_19.height, "(ATL terminó 2024 con record 8-9, sin playoffs)")

filas de ATL en 'semana 19' marcada REG: 59 (ATL terminó 2024 con record 8-9, sin playoffs)


Por eso `cargar_depth_charts_historico()` acota la semana a la última real de temporada regular (17 hasta 2020, 18 desde 2021), no solo `game_type=="REG"`. La función también traduce los códigos de equipo de la época (`OAK`, `SD`) a los actuales (`LV`, `LAC`), que son los que usa la tabla de estadísticas (ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)).

### Las dos funciones, con los filtros aplicados

In [9]:
hist = data.cargar_depth_charts_historico([2024])
print("=== historico 2024 ===")
print("filas:", hist.shape[0], "| semanas:", sorted(hist["week"].unique()))
print(hist[(hist["club_code"] == "PHI") & (hist["week"] == 1)].sort_values("depth_team"))

print()
actual = data.cargar_depth_charts_actual([2026])
print("=== actual 2026 (temporada en curso) ===")
ultima = actual[actual["team"] == "PHI"]["dt"].max()
print("ultimo snapshot disponible para PHI:", ultima)
print(actual[(actual["team"] == "PHI") & (actual["dt"] == ultima)].sort_values("pos_rank"))

=== historico 2024 ===
filas: 3051 | semanas: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]
      season  week club_code     gsis_id      full_name  depth_team
1936    2024     1       PHI  00-0037741   Jahan Dotson           1
1937    2024     1       PHI  00-0035676     A.J. Brown           1
1938    2024     1       PHI  00-0036912  DeVonta Smith           1
1934    2024     1       PHI  00-0037132  Britain Covey           2
1935    2024     1       PHI  00-0039236  Johnny Wilson           2

=== actual 2026 (temporada en curso) ===
ultimo snapshot disponible para PHI: 2026-09-29 13:59:18+00:00
                           dt team     gsis_id       player_name  pos_rank
167 2026-09-29 13:59:18+00:00  PHI  00-0036912     DeVonta Smith         1
168 2026-09-29 13:59:18+00:00  PHI  00-0038393 

## Qué se agregó después

`data.py` tiene hoy dos funciones más, construidas en notebooks posteriores:

- `cargar_depth_charts_unificado`: combina las dos alineaciones (`_historico` y `_actual`) en una sola
  tabla desde 2016; se explica en [`1.3_unificacion_alineaciones.ipynb`](1.3_unificacion_alineaciones.ipynb).
- `identificar_qb_titular`: encuentra al mariscal de campo titular de cada equipo y semana; se usa en
  [`3.1_ingenieria_features.ipynb`](3.1_ingenieria_features.ipynb).

Las variables que se calculan a partir de estas tablas (promedios de temporada, de los últimos N
juegos y de carrera) viven en `features.py`, y el notebook siguiente las explica.

Código fuente: [`../src/data.py`](../src/data.py) · Hallazgos completos:
[`../../../docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)

Siguiente: [`1.4_promedios_sin_fuga.ipynb`](1.4_promedios_sin_fuga.ipynb)